# Data types and safe conversion · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 09. Data types and safe conversion

Choose a useful data type and identify values that cannot be converted.

Before you start: You can select a column and create a new column.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#dtypes

### M09C01. A number and its text are different

A data type tells pandas what a column contains and which operations make sense. Quotation marks make "20" text. `astype()` converts values when the requested conversion is possible.

**Read:** What do the quotation marks around the ages tell Python?

Answer: These values are strings, even though they contain digits.

**Predict:** What will the next-year ages be?

Answer: 21, 22, and 20.

**Explain:** Why convert before adding 1?

Answer: We want numeric addition; text values are not numeric ages yet.



In [ ]:
# Predict before running: What will the next-year ages be?
ages = pd.Series(["20", "21", "19"], dtype="string")
result = ages.astype("int64")
print("Before:", ages.dtype)
print("After:", result.dtype)
print("Next year:", (result + 1).tolist())

**Your turn:** Start again with the same text ages. Convert them to numbers and calculate their mean in `result`.

For the check, keep these result names: `result`.

**Hint:** Convert the Series first; then call `.mean()`.

**Watch for this:** Do not assume a column is numeric just because its displayed values look like numbers.

In [ ]:
# Change this code to complete the task.
# Question: Start again with the same text ages. Convert them to numbers and calculate their mean in `result`.
ages = pd.Series(["20", "21", "19"], dtype="string")
result = ages.astype("int64").mean()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == 20.0
print("Code checks passed. Review the written explanation as well.")

The mean is 20.0. The conversion makes arithmetic meaningful.

**Quick check:** Which expression requests a numeric type?

1. ages.astype("int64")
2. ages.astype("string")
3. str(ages)

Answer: 1. int64 is an integer data type; string is text.

**Explain what you learned:** Check types before doing arithmetic.

### M09C02. Convert messy numbers and keep the reason

`pd.to_numeric()` parses values as numbers. With `errors="coerce"`, values it cannot parse become missing. Preserve the source so you can distinguish an invalid label from a value that was already missing.

**Read:** What does `errors="coerce"` ask pandas to do?

Answer: Replace a value that cannot be parsed as a number with a missing value.

**Predict:** Which row is marked as invalid text?

Answer: Only row 1, containing "absent". Row 2 was already missing.

**Explain:** Why should we keep the original values?

Answer: Conversion alone loses the distinction between invalid text and an originally missing value; "absent" may also have a special policy meaning.



In [ ]:
# Predict before running: Which row is marked as invalid text?
source = pd.Series(["88", "absent", None, "72"], dtype="string")
numbers = pd.to_numeric(source, errors="coerce")
invalid = source.notna() & numbers.isna()
result = pd.DataFrame({"Original": source, "Number": numbers, "Invalid_text": invalid})
print(result)

**Your turn:** Change "absent" to "90" in the source. Rebuild the table and count the missing numbers.

For the check, keep these result names: `result`.

**Hint:** Use `.isna().sum()` after the conversion.

**Watch for this:** Do not silently turn every conversion failure into zero.

In [ ]:
# Change this code to complete the task.
# Question: Change "absent" to "90" in the source. Rebuild the table and count the missing numbers.
source = pd.Series(["88", "90", None, "72"], dtype="string")
numbers = pd.to_numeric(source, errors="coerce")
invalid = source.notna() & numbers.isna()
result = pd.DataFrame({"Original": source, "Number": numbers, "Invalid_text": invalid})
print(result)
print("Missing numbers:", numbers.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Number"].isna().sum() == 1
assert not result["Invalid_text"].any()
print("Code checks passed. Review the written explanation as well.")

Only the original None remains missing; there are no invalid text values.

**Quick check:** Does converting "absent" to a missing value mean the score was zero?

1. Yes, missing and zero are identical
2. No; a grading rule must determine what absent means
3. Yes, pandas automatically applies grading rules

Answer: 2. Parsing is a technical operation. It does not decide the meaning or grading treatment of absence.

**Explain what you learned:** Audit conversion failures before deciding how to handle them.

### M09C03. Integers that can be missing

Pandas nullable integer type `Int64` can store whole numbers and `pd.NA`. Its capital I matters. Comparisons can also be missing, so decide how missing comparisons should behave in a filter.

**Read:** What does the capital I in `Int64` select?

Answer: Pandas nullable integer type, which supports a missing-value marker.

**Predict:** What happens to the missing age when 1 is added?

Answer: It remains missing; pandas does not invent an age.

**Explain:** Why is None not an age of zero?

Answer: It means the age was not recorded, while zero would be an actual number.



In [ ]:
# Predict before running: What happens to the missing age when 1 is added?
ages = pd.Series([20, None, 22], dtype="Int64")
result = ages + 1
print(result)
print("Known ages:", ages.notna().sum())

**Your turn:** Start with the same ages and keep only known ages of at least 21. Store the filtered Series in `result`.

For the check, keep these result names: `result`.

**Hint:** Build `ages >= 21`; use `.fillna(False)` on that mask.

**Watch for this:** Do not write `if pd.NA:`; its truth value is ambiguous. Use `.isna()` or `.notna()`.

In [ ]:
# Change this code to complete the task.
# Question: Start with the same ages and keep only known ages of at least 21. Store the filtered Series in `result`.
ages = pd.Series([20, None, 22], dtype="Int64")
mask = (ages >= 21).fillna(False)
result = ages[mask]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [22]
assert result.index.tolist() == [2]
print("Code checks passed. Review the written explanation as well.")

Only 22 remains. The explicit False says an unknown age does not satisfy this selection rule.

**Quick check:** Which value remains missing after adding 1?

1. 20
2. None / pd.NA
3. 22

Answer: 2. Arithmetic preserves the unknown value.

**Explain what you learned:** Use nullable types when missing values are legitimate.

### M09C04. Categories and an explicit order

A category stores values drawn from a defined set, such as programs or rating bands. An ordered category has a meaningful sequence that you choose. Sorting follows that sequence instead of alphabetical order.

**Read:** Where is the rating order defined?

Answer: In the category list ["Low", "Medium", "High"].

**Predict:** Which value appears first after sorting?

Answer: Low, followed by the other Low, then Medium and High.

**Explain:** Why might alphabetical sorting be wrong here?

Answer: Alphabetical sorting puts High first, which does not represent the intended rating order.



In [ ]:
# Predict before running: Which value appears first after sorting?
ratings = pd.Series(["High", "Low", "Medium", "Low"], dtype="string")
rating_type = pd.CategoricalDtype(["Low", "Medium", "High"], ordered=True)
result = ratings.astype(rating_type).sort_values()
print(result)
print("Order:", result.cat.categories.tolist())

**Your turn:** Use students(). Make Program an ordered category with IT before AI, then sort by Program and Student_ID. Store the sorted Student_ID and Program columns in the DataFrame result.

For the check, keep these result names: `result`.

**Hint:** Create a `pd.CategoricalDtype(["IT", "AI"], ordered=True)`.

**Watch for this:** A value outside a declared category list becomes missing when converted; check the result.

In [ ]:
# Change this code to complete the task.
# Question: Use students(). Make Program an ordered category with IT before AI, then sort by Program and Student_ID. Store the sorted Student_ID and Program columns in the DataFrame result.
df = students()
program_type = pd.CategoricalDtype(["IT", "AI"], ordered=True)
df["Program"] = df["Program"].astype(program_type)
result = df.sort_values(["Program", "Student_ID"])[["Student_ID", "Program"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Program"].iloc[:6].tolist() == ["IT"] * 6
assert result.iloc[0]["Student_ID"] == "S002"
print("Code checks passed. Review the written explanation as well.")

All six IT students appear before the six AI students; each program is ordered by ID.

**Quick check:** What does ordered=True add?

1. An explicit meaningful order
2. Automatic numeric grades
3. A rule that removes duplicates

Answer: 1. The category sequence becomes the ordering used by sorting and comparisons.

**Explain what you learned:** Define category meaning and order deliberately.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates